# DevGen — Hybrid StyleGAN-Attention Conditional GAN (Production v3)

**Best-in-class** text-conditional handwriting generator combining:
- StyleGAN2-style weight modulation with noise injection
- Spatial text encoding with bottleneck self-attention
- CTC-guided recognizer feedback with curriculum scheduling
- DiffAugment for data-efficient training
- Exponential Moving Average (EMA) generator for stable inference
- R1 gradient penalty for discriminator regularization
- Anti-collapse monitoring with automatic recovery

### Kaggle Setup:
1. **GPU**: Accelerator → **GPU T4 x2**
2. **Internet**: **ON**
3. **Add Input**: Upload previous checkpoint dataset if resuming
4. Click **Run All**.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 1: Environment Setup & Dependency Installation
# ══════════════════════════════════════════════════════════════════
!pip install -q datasets Levenshtein

import os, time, sys, copy, math, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from datasets import load_dataset
from PIL import Image
import numpy as np

# ─── CONFIGURATION ───
LATENT_DIM     = 128
EMBED_DIM      = 128
STYLE_DIM      = 256
IMG_H, IMG_W   = 64, 256
BATCH_SIZE     = 64
MAX_TXT_LEN    = 16
TOTAL_TARGET_STEPS = 50000
RECOGNIZER_PRETRAIN_STEPS = 5000
CHECKPOINT_INTERVAL = 5000
SAMPLE_INTERVAL = 500
LOG_INTERVAL    = 100
MAX_TIME        = 39000       # ~10.8 hours Kaggle safety
EMA_DECAY       = 0.9995
R1_GAMMA        = 10.0
R1_INTERVAL     = 16          # Apply R1 every N discriminator steps
N_CRITIC        = 1           # D steps per G step
LR_G            = 1e-4
LR_D            = 2e-4
LR_R            = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── CHECKPOINT RESUME FILENAME ───
CHECKPOINT_FILENAME = "hybrid_step_35000.pt"  # Change to match your latest!

os.makedirs("stylegan_attn_samples", exist_ok=True)
os.makedirs("stylegan_attn_checkpoints", exist_ok=True)

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def pad_to_size(img, target_w=IMG_W, target_h=IMG_H):
    """Resize image maintaining aspect ratio, centered on white canvas with background normalization."""
    w, h = img.size
    if w == 0 or h == 0:
        return Image.new("L", (target_w, target_h), 255)
    
    # 1. Background Normalization:
    # Convert image to numpy array to estimate background illumination and scale contrast
    np_img = np.array(img.convert("L"))
    # The 90th percentile represents the background white value of the paper
    bg_val = max(1.0, float(np.percentile(np_img, 90)))
    # Stretch contrast so background becomes pure white (255)
    np_img_clean = np.clip(np_img.astype(float) * (255.0 / bg_val), 0, 255).astype(np.uint8)
    img_clean = Image.fromarray(np_img_clean)
    
    # 2. Resizing with aspect ratio preservation
    ratio = min(target_w / w, target_h / h)
    new_w, new_h = max(1, int(w * ratio)), max(1, int(h * ratio))
    img_resized = img_clean.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    # 3. Create pure white canvas and paste
    padded = Image.new("L", (target_w, target_h), 255)
    padded.paste(img_resized, ((target_w - new_w) // 2, (target_h - new_h) // 2))
    return padded
print("Loading dataset from HuggingFace...")
hf_ds = load_dataset("c3rl/IIIT-INDIC-HW-WORDS-Hindi", split="train+validation+test")

# Build vocabulary from valid samples only
all_texts = hf_ds["text"]
valid_indices = [i for i, text in enumerate(all_texts) if text and len(text.strip()) > 0]
raw_vocab = "".join([all_texts[i] for i in valid_indices])
vocab = sorted(list(set(raw_vocab)))
char_to_idx = {char: idx + 1 for idx, char in enumerate(vocab)}
idx_to_char = {idx + 1: char for idx, char in enumerate(vocab)}
VOCAB_SIZE = len(char_to_idx) + 1  # +1 for padding/blank (index 0)

class ConditionalWordDataset(Dataset):
    def __init__(self, ds, valid_idx, char_map, max_len):
        self.ds = ds
        self.valid_idx = valid_idx
        self.char_map = char_map
        self.max_len = max_len
        self.tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    
    def __len__(self):
        return len(self.valid_idx)
    
    def __getitem__(self, i):
        try:
            row = self.ds[self.valid_idx[i]]
            img = self.tf(pad_to_size(row["image"].convert("L")))
            tokens = [self.char_map[c] for c in row["text"] if c in self.char_map][:self.max_len]
            t_len = len(tokens)
            if t_len == 0:
                tokens = [1]  # fallback
                t_len = 1
            
            # Left-aligned tokens for CTC loss targets (index 0 blank constraint)
            ctc_tokens = tokens + [0] * (self.max_len - t_len)
            
            # Centered tokens for Spatial Text Encoder layout alignment
            left_pad = (self.max_len - t_len) // 2
            right_pad = self.max_len - t_len - left_pad
            gen_tokens = [0] * left_pad + tokens + [0] * right_pad
            
            return img, torch.tensor(gen_tokens, dtype=torch.long), torch.tensor(ctc_tokens, dtype=torch.long), torch.tensor(t_len, dtype=torch.long)
        except Exception:
            img = torch.zeros(1, IMG_H, IMG_W)
            dummy_tokens = [1] + [0] * (self.max_len - 1)
            return img, torch.tensor(dummy_tokens, dtype=torch.long), torch.tensor(dummy_tokens, dtype=torch.long), torch.tensor(1, dtype=torch.long)

dataset = ConditionalWordDataset(hf_ds, valid_indices, char_to_idx, MAX_TXT_LEN)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=2, pin_memory=True, persistent_workers=True
)
print(f"Dataset ready: {len(valid_indices)} samples, vocab_size={VOCAB_SIZE} ({len(vocab)} unique chars)")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 3: DiffAugment & Neural Architecture Definitions
# ══════════════════════════════════════════════════════════════════

# ─── DIFF_AUGMENTATION ENGINE ───
def apply_diff_augment(x):
    """Differentiable augmentation: brightness, contrast, translation, cutout."""
    B, C, H, W = x.size()
    # Color jitter
    brightness = (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.25
    x = x + brightness
    contrast = torch.rand(B, 1, 1, 1, device=x.device) * 0.35 + 0.8
    mean = torch.mean(x, dim=[2, 3], keepdim=True)
    x = (x - mean) * contrast + mean
    
    # Translation
    max_t_x, max_t_y = 14.0 / W, 3.0 / H
    tx = (torch.rand(B, device=x.device) * 2.0 - 1.0) * max_t_x
    ty = (torch.rand(B, device=x.device) * 2.0 - 1.0) * max_t_y
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 1, 1] = 1.0, 1.0
    theta[:, 0, 2], theta[:, 1, 2] = tx, ty
    grid = F.affine_grid(theta, x.size(), align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='reflection', align_corners=False)
    
    # Cutout (random rectangular mask)
    if False:  # Disabled cutout to preserve Devanagari character stroke connectivity
        cut_h = int(H * 0.15)
        cut_w = int(W * 0.15)
        cy = torch.randint(cut_h, H - cut_h, (B,))
        cx = torch.randint(cut_w, W - cut_w, (B,))
        for b in range(B):
            x[b, :, cy[b]-cut_h:cy[b]+cut_h, cx[b]-cut_w:cx[b]+cut_w] = 0
    
    return torch.clamp(x, -1.0, 1.0)

# ─── ADVANCED MODULES ───
class LightweightBottleneckAttention(nn.Module):
    """Memory-efficient self-attention with bottleneck projection."""
    def __init__(self, in_channels):
        super().__init__()
        neck = max(in_channels // 8, 1)
        self.query = nn.Conv2d(in_channels, neck, 1)
        self.key   = nn.Conv2d(in_channels, neck, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        B, C, H, W = x.size()
        q = self.query(x).view(B, -1, H * W).permute(0, 2, 1)
        k = self.key(x).view(B, -1, H * W)
        v = self.value(x).view(B, -1, H * W)
        # Scaled dot-product attention
        scale = q.size(-1) ** -0.5
        attn = torch.softmax(torch.bmm(q, k) * scale, dim=-1)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return x + self.gamma * out

class StyleModulatedConv2d(nn.Module):
    """StyleGAN2-style weight demodulation convolution."""
    def __init__(self, in_channels, out_channels, kernel_size, style_dim, demodulate=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.demodulate = demodulate
        self.padding = kernel_size // 2
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.weight, a=0.2, mode='fan_in', nonlinearity='leaky_relu')
        self.style_proj = nn.Linear(style_dim, in_channels)
        self.style_proj.bias.data.fill_(1.0)
    
    def forward(self, x, style):
        B, C, H, W = x.size()
        s = self.style_proj(style).reshape(B, 1, C, 1, 1)
        w = self.weight.unsqueeze(0) * s
        if self.demodulate:
            demod = torch.rsqrt(w.pow(2).sum(dim=[2, 3, 4]) + 1e-8)
            w = w * demod.reshape(B, self.out_channels, 1, 1, 1)
        x = x.reshape(1, B * C, H, W)
        w = w.reshape(B * self.out_channels, self.in_channels, self.kernel_size, self.kernel_size)
        return F.conv2d(x, w, padding=self.padding, groups=B).reshape(B, self.out_channels, H, W)

class NoiseInjection(nn.Module):
    """Per-pixel noise injection for stochastic variation (ink texture)."""
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        noise = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device)
        return x + self.weight * noise

# ─── NEURAL ARCHITECTURE SUB-SYSTEMS ───
class SpatialTextEncoder(nn.Module):
    """Bidirectional GRU text encoder producing spatial feature maps."""
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, embed_dim // 2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(embed_dim, 512)
    
    def forward(self, text):
        embedded = self.embedding(text)
        outputs, hidden = self.gru(embedded)
        spatial_feats = self.fc(outputs).permute(0, 2, 1).unsqueeze(2)  # [B, 512, 1, seq_len]
        global_feat = torch.cat([hidden[0], hidden[1]], dim=-1)         # [B, embed_dim]
        return spatial_feats, global_feat

class StyleMappingNetwork(nn.Module):
    """8-layer MLP mapping z to style space (with pixel-norm on input)."""
    def __init__(self, latent_dim, style_dim):
        super().__init__()
        layers = []
        for i in range(6):
            in_dim = latent_dim if i == 0 else style_dim
            layers.extend([nn.Linear(in_dim, style_dim), nn.LeakyReLU(0.2, True)])
        self.net = nn.Sequential(*layers)
    
    def forward(self, z):
        # Pixel normalization of input latent
        z = z / (torch.sqrt(torch.mean(z ** 2, dim=1, keepdim=True)) + 1e-8)
        return self.net(z)

class ModulatedUpsampleBlock(nn.Module):
    def __init__(self, in_chan, out_chan, style_dim):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = StyleModulatedConv2d(in_chan, out_chan, 3, style_dim)
        self.noise = NoiseInjection()
        self.lrelu = nn.LeakyReLU(0.2, True)
    
    def forward(self, x, style):
        x = self.up(x)
        x = self.conv(x, style)
        x = self.noise(x)
        return self.lrelu(x)

class ModulatedRefineBlock(nn.Module):
    def __init__(self, chan, style_dim):
        super().__init__()
        self.conv1 = StyleModulatedConv2d(chan, chan, 3, style_dim)
        self.noise1 = NoiseInjection()
        self.lrelu = nn.LeakyReLU(0.2, True)
        self.conv2 = StyleModulatedConv2d(chan, chan, 3, style_dim)
        self.noise2 = NoiseInjection()
    
    def forward(self, x, style):
        residual = x
        x = self.noise1(self.conv1(x, style))
        x = self.lrelu(x)
        x = self.noise2(self.conv2(x, style))
        return residual + x

class AdvancedStyleGANAttentionGenerator(nn.Module):
    def __init__(self, style_dim):
        super().__init__()
        self.expand_height = nn.ConvTranspose2d(512, 512, kernel_size=(4, 1), stride=(4, 1))
        self.bottleneck_attention = LightweightBottleneckAttention(512)
        self.up1 = ModulatedUpsampleBlock(512, 256, style_dim)
        self.up2 = ModulatedUpsampleBlock(256, 128, style_dim)
        self.up3 = ModulatedUpsampleBlock(128, 64, style_dim)
        self.up4 = ModulatedUpsampleBlock(64, 32, style_dim)
        self.refine = ModulatedRefineBlock(32, style_dim)
        self.out = nn.Sequential(nn.Conv2d(32, 1, 3, 1, 1), nn.Tanh())
        
        # SOTA StyleGAN2 Multi-Scale RGB Skip Projections
        self.to_rgb1 = nn.Conv2d(256, 1, 1)
        self.to_rgb2 = nn.Conv2d(128, 1, 1)
        self.to_rgb3 = nn.Conv2d(64, 1, 1)
        self.to_rgb4 = nn.Conv2d(32, 1, 1)
    
    def forward(self, spatial_feats, style):
        x = self.expand_height(spatial_feats)
        x = self.bottleneck_attention(x)
        
        # Resolution 8x32
        x = self.up1(x, style)
        rgb = self.to_rgb1(x)
        
        # Resolution 16x64
        x = self.up2(x, style)
        rgb = F.interpolate(rgb, scale_factor=2, mode='bilinear', align_corners=False) + self.to_rgb2(x)
        
        # Resolution 32x128
        x = self.up3(x, style)
        rgb = F.interpolate(rgb, scale_factor=2, mode='bilinear', align_corners=False) + self.to_rgb3(x)
        
        # Resolution 64x256
        x = self.up4(x, style)
        rgb = F.interpolate(rgb, scale_factor=2, mode='bilinear', align_corners=False) + self.to_rgb4(x)
        
        # Refine final resolution feature maps
        x = self.refine(x, style)
        out = self.out(x)
        
        # Final image is the sum of refined features and multi-scale rgb projections
        return torch.tanh(out + rgb)

class SNSpectralPatchDiscriminator(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.block1 = nn.Sequential(nn.utils.spectral_norm(nn.Conv2d(1, 64, 4, 2, 1)), nn.LeakyReLU(0.2, True))
        self.block2 = nn.Sequential(nn.utils.spectral_norm(nn.Conv2d(64, 128, 4, 2, 1)), nn.LeakyReLU(0.2, True))
        self.block3 = nn.Sequential(nn.utils.spectral_norm(nn.Conv2d(128, 256, 4, 2, 1)), nn.LeakyReLU(0.2, True))
        self.block4 = nn.Sequential(nn.utils.spectral_norm(nn.Conv2d(256, 512, 4, 2, 1)), nn.LeakyReLU(0.2, True))
        self.uncond_out = nn.utils.spectral_norm(nn.Conv2d(512, 1, 3, 1, 1))
        self.text_proj = nn.utils.spectral_norm(nn.Linear(embed_dim, 512))
    
    def forward(self, x, text_embed):
        f = self.block4(self.block3(self.block2(self.block1(x))))
        uncond = self.uncond_out(f)
        t = self.text_proj(text_embed).view(-1, 512, 1, 1)
        return uncond + torch.sum(f * t, dim=1, keepdim=True)

class TextRecognizer(nn.Module):
    """Lightweight CTC-based text recognizer for generator guidance."""
    def __init__(self, vocab_size):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1))
        )
        self.rnn = nn.GRU(256 * 4, 128, bidirectional=True, batch_first=True, num_layers=2, dropout=0.1)
        self.fc = nn.Linear(256, vocab_size)
    
    def forward(self, x):
        f = self.conv(x)
        b, c, h, w = f.size()
        f = f.permute(0, 3, 1, 2).contiguous().view(b, w, c * h)
        o, _ = self.rnn(f)
        return self.fc(o).permute(1, 0, 2)  # [T, B, vocab_size]

print("All architectures defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 4: Model Instantiation, EMA, & Auto-Resume
# ══════════════════════════════════════════════════════════════════

# ─── Instantiate all networks ───
E = SpatialTextEncoder(VOCAB_SIZE, EMBED_DIM).to(DEVICE)
mapping_net = StyleMappingNetwork(LATENT_DIM, STYLE_DIM).to(DEVICE)
G = AdvancedStyleGANAttentionGenerator(STYLE_DIM).to(DEVICE)
D = SNSpectralPatchDiscriminator().to(DEVICE)
R = TextRecognizer(VOCAB_SIZE).to(DEVICE)

# ─── EMA Generator ───
G_ema = copy.deepcopy(G).eval()
for p in G_ema.parameters():
    p.requires_grad_(False)

def update_ema(ema_model, model, decay):
    with torch.no_grad():
        for ema_p, model_p in zip(ema_model.parameters(), model.parameters()):
            ema_p.data.mul_(decay).add_(model_p.data, alpha=1.0 - decay)

# ─── Optimizers with TTUR ───
opt_G = optim.Adam(
    list(G.parameters()) + list(E.parameters()) + list(mapping_net.parameters()),
    lr=LR_G, betas=(0.0, 0.99)
)
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.0, 0.99))
opt_R = optim.Adam(R.parameters(), lr=LR_R)

ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

# Compute CTC input_lengths based on the recognizer's temporal output width
# For input W=256: conv pools horizontally by (2,2,1,1) => W/4 = 64
CTC_INPUT_LENGTH = IMG_W // 4
input_lengths = torch.full(size=(BATCH_SIZE,), fill_value=CTC_INPUT_LENGTH, dtype=torch.long, device=DEVICE)

# ─── AUTO-RESUME: Scan /kaggle/input/ and local checkpoints ───
import shutil

def resolve_checkpoint_path(filename):
    """Search for checkpoint in multiple locations."""
    local = os.path.join("stylegan_attn_checkpoints", filename)
    if os.path.exists(local): return local
    if os.path.exists(filename): return filename
    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            if filename in files:
                return os.path.join(root, filename)
    # Also try finding the highest step checkpoint
    ckpt_dir = "stylegan_attn_checkpoints"
    if os.path.exists(ckpt_dir):
        files = [f for f in os.listdir(ckpt_dir) if f.startswith("hybrid_step_") and f.endswith(".pt")]
        if files:
            latest = sorted(files, key=lambda x: int(x.split('_')[2].split('.')[0]))[-1]
            return os.path.join(ckpt_dir, latest)
    return None

start_step = 1
ckpt_path = resolve_checkpoint_path(CHECKPOINT_FILENAME)
if ckpt_path:
    print(f"📦 Checkpoint found: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    
    # Load G with strict=False to support new ToRGB projection parameters
    try:
        G.load_state_dict(ckpt['generator_state_dict'], strict=True)
        print("✅ Loaded generator in strict mode.")
    except Exception as e:
        print(f"⚠️ Strict loading failed, loading with strict=False to accommodate ToRGB layers: {e}")
        G.load_state_dict(ckpt['generator_state_dict'], strict=False)
        # Initialize new layers to zero so they don't corrupt the checkpoint features initially
        with torch.no_grad():
            for name, param in G.named_parameters():
                if 'to_rgb' in name:
                    param.zero_()
        print("✅ Initialized upgraded ToRGB layer parameters to zero.")
        
    D.load_state_dict(ckpt['discriminator_state_dict'])
    E.load_state_dict(ckpt['encoder_state_dict'])
    mapping_net.load_state_dict(ckpt['mapping_state_dict'])
    R.load_state_dict(ckpt['recognizer_state_dict'])
    
    # Load EMA if available
    if 'ema_state_dict' in ckpt:
        try:
            G_ema.load_state_dict(ckpt['ema_state_dict'], strict=True)
        except Exception:
            G_ema.load_state_dict(ckpt['ema_state_dict'], strict=False)
            with torch.no_grad():
                for name, param in G_ema.named_parameters():
                    if 'to_rgb' in name:
                        param.zero_()
    else:
        G_ema.load_state_dict(G.state_dict())
    
    # Restore optimizer states
    if 'opt_G_state_dict' in ckpt: opt_G.load_state_dict(ckpt['opt_G_state_dict'])
    if 'opt_D_state_dict' in ckpt: opt_D.load_state_dict(ckpt['opt_D_state_dict'])
    if 'opt_R_state_dict' in ckpt: opt_R.load_state_dict(ckpt['opt_R_state_dict'])
    
    start_step = ckpt['step'] + 1
    print(f"✅ Resumed from step {start_step - 1}")
    del ckpt
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print(f"🚀 No checkpoint found. Starting fresh training.")
    G_ema.load_state_dict(G.state_dict())

# Count parameters
total_params = sum(p.numel() for p in G.parameters()) + sum(p.numel() for p in E.parameters()) + sum(p.numel() for p in mapping_net.parameters())
print(f"Generator+Encoder total params: {total_params/1e6:.1f}M")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters())/1e6:.1f}M")
print(f"Recognizer params: {sum(p.numel() for p in R.parameters())/1e6:.1f}M")
print(f"Training from step {start_step} to {TOTAL_TARGET_STEPS}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5: Training Loop
# ══════════════════════════════════════════════════════════════════

step = start_step
fixed_batch = next(iter(loader))
fixed_z = torch.randn(16, LATENT_DIM, device=DEVICE)
fixed_tokens = fixed_batch[1][:16].to(DEVICE)  # gen_tokens (centered)
break_execution = False
start_time = time.time()

# Loss tracking for anti-collapse detection
d_loss_history = []
g_loss_history = []

G.train(); D.train(); R.train(); mapping_net.train(); E.train()
print(f"▶️ Training started. Time limit: {MAX_TIME/3600:.1f}h | Target: {TOTAL_TARGET_STEPS} steps")

for epoch in range(9999):
    if break_execution: break
    for real_imgs, gen_tokens, ctc_tokens, lens in loader:
        elapsed = time.time() - start_time
        if step > TOTAL_TARGET_STEPS or elapsed > MAX_TIME:
            if elapsed > MAX_TIME:
                print(f"\n⏰ Time limit reached ({elapsed/3600:.1f}h).")
            else:
                print(f"\n✅ Completed {TOTAL_TARGET_STEPS} steps successfully!")
            break_execution = True
            break

        real_imgs = real_imgs.to(DEVICE, non_blocking=True)
        gen_tokens = gen_tokens.to(DEVICE, non_blocking=True)
        ctc_tokens = ctc_tokens.to(DEVICE, non_blocking=True)
        lens = lens.to(DEVICE, non_blocking=True)
        
        # ─── RECOGNIZER PRE-TRAINING PHASE ───
        if step <= RECOGNIZER_PRETRAIN_STEPS:
            opt_R.zero_grad(set_to_none=True)
            r_real = torch.log_softmax(R(real_imgs), dim=-1)
            loss_R = ctc_loss_fn(r_real, ctc_tokens, input_lengths, lens)
            if not (torch.isnan(loss_R) or torch.isinf(loss_R)):
                loss_R.backward()
                nn.utils.clip_grad_norm_(R.parameters(), 1.0)
                opt_R.step()
            
            if step % LOG_INTERVAL == 0:
                print(f"[Pretrain R] Step {step}/{RECOGNIZER_PRETRAIN_STEPS} | CTC Loss: {loss_R.item():.4f}")
            
            if step % CHECKPOINT_INTERVAL == 0 or step == RECOGNIZER_PRETRAIN_STEPS:
                cp = f"stylegan_attn_checkpoints/hybrid_step_{step}.pt"
                torch.save({
                    'step': step,
                    'generator_state_dict': G.state_dict(),
                    'ema_state_dict': G_ema.state_dict(),
                    'discriminator_state_dict': D.state_dict(),
                    'encoder_state_dict': E.state_dict(),
                    'mapping_state_dict': mapping_net.state_dict(),
                    'recognizer_state_dict': R.state_dict(),
                    'opt_G_state_dict': opt_G.state_dict(),
                    'opt_D_state_dict': opt_D.state_dict(),
                    'opt_R_state_dict': opt_R.state_dict(),
                    'char_to_idx': char_to_idx,
                    'idx_to_char': idx_to_char,
                    'vocab': vocab
                }, cp)
                print(f"📑 Pretrain Checkpoint saved: {cp}")
            
            step += 1
            continue
        
        # Forward Curriculum CTC guidance: start weak (untrained OCR), ramp up as OCR improves
        progress = min(1.0, (step - 1) / TOTAL_TARGET_STEPS)
        guidance_weight = min(0.80, 0.05 + progress * 0.75)

        # ─── 1. DISCRIMINATOR ───
        for d_step in range(N_CRITIC):
            opt_D.zero_grad(set_to_none=True)
            
            with torch.no_grad():
                spatial_feats, global_feat = E(gen_tokens)
                z = torch.randn(BATCH_SIZE, LATENT_DIM, device=DEVICE)
                style = mapping_net(z)
                fake_imgs = G(spatial_feats, style)
            
            d_real = D(apply_diff_augment(real_imgs), global_feat.detach())
            d_fake = D(apply_diff_augment(fake_imgs), global_feat.detach())
            
            # Hinge loss
            loss_D = torch.mean(F.relu(1.0 - d_real)) + torch.mean(F.relu(1.0 + d_fake))
            
            # R1 gradient penalty (applied periodically)
            if step % R1_INTERVAL == 0:
                real_imgs_gp = real_imgs.detach().requires_grad_(True)
                d_real_gp = D(real_imgs_gp, global_feat.detach())
                grad_real = torch.autograd.grad(
                    outputs=d_real_gp.mean(dim=[2, 3]).sum(), inputs=real_imgs_gp, create_graph=True
                )[0]
                r1_penalty = grad_real.pow(2).reshape(grad_real.size(0), -1).sum(dim=1).mean()
                loss_D = loss_D + (R1_GAMMA / 2.0) * r1_penalty * R1_INTERVAL
            
            loss_D.backward()
            nn.utils.clip_grad_norm_(D.parameters(), 1.0)
            opt_D.step()

        # ─── 2. RECOGNIZER (Train on both real and frozen-generator fakes) ───
        opt_R.zero_grad(set_to_none=True)
        r_real = torch.log_softmax(R(real_imgs), dim=-1)
        loss_R_real = ctc_loss_fn(r_real, ctc_tokens, input_lengths, lens)
        
        with torch.no_grad():
            z2 = torch.randn(BATCH_SIZE, LATENT_DIM, device=DEVICE)
            spatial_feats_r, _ = E(gen_tokens)
            fake_for_R = G(spatial_feats_r, mapping_net(z2))
        r_fake = torch.log_softmax(R(fake_for_R), dim=-1)
        loss_R_fake = ctc_loss_fn(r_fake, ctc_tokens, input_lengths, lens)
        
        loss_R = 0.5 * loss_R_real + 0.5 * loss_R_fake
        if not (torch.isnan(loss_R) or torch.isinf(loss_R)):
            loss_R.backward()
            nn.utils.clip_grad_norm_(R.parameters(), 1.0)
            opt_R.step()

        # ─── 3. GENERATOR (GAN loss + CTC guidance) ───
        opt_G.zero_grad(set_to_none=True)
        spatial_feats, global_feat = E(gen_tokens)
        z = torch.randn(BATCH_SIZE, LATENT_DIM, device=DEVICE)
        style = mapping_net(z)
        gen_imgs = G(spatial_feats, style)
        
        g_logits = D(apply_diff_augment(gen_imgs), global_feat)
        gan_loss = -torch.mean(g_logits)
        
        r_gen = torch.log_softmax(R(gen_imgs), dim=-1)
        guidance_loss = ctc_loss_fn(r_gen, ctc_tokens, input_lengths, lens)
        
        total_G_loss = gan_loss + guidance_weight * guidance_loss
        
        if not (torch.isnan(total_G_loss) or torch.isinf(total_G_loss)):
            total_G_loss.backward()
            # Gradient clipping (SOTA training stability)
            nn.utils.clip_grad_norm_(
                list(G.parameters()) + list(E.parameters()) + list(mapping_net.parameters()), 1.0
            )
            opt_G.step()
        
        # Update EMA generator
        update_ema(G_ema, G, EMA_DECAY)

        # ─── LOGGING ───
        if step % LOG_INTERVAL == 0:
            rate = (step - start_step + 1) / max(elapsed, 1)
            eta_h = (TOTAL_TARGET_STEPS - step) / max(rate, 0.01) / 3600
            d_loss_val = loss_D.item()
            g_loss_val = gan_loss.item()
            d_loss_history.append(d_loss_val)
            g_loss_history.append(g_loss_val)
            
            # Anti-collapse detection
            collapse_warning = ""
            if len(d_loss_history) > 20:
                recent_d = d_loss_history[-20:]
                if all(abs(d) < 0.01 for d in recent_d):
                    collapse_warning = " ⚠️ POSSIBLE COLLAPSE"
            
            print(
                f"Step {step:05d}/{TOTAL_TARGET_STEPS} | "
                f"D: {d_loss_val:.4f} | G: {g_loss_val:.4f} | "
                f"λ: {guidance_weight:.3f} | CTC_R: {loss_R_real.item():.3f} | "
                f"{rate:.1f} st/s | ETA: {eta_h:.1f}h{collapse_warning}"
            )

        # ─── SAMPLE GENERATION (EMA generator) ───
        if step % SAMPLE_INTERVAL == 0 or step == TOTAL_TARGET_STEPS:
            with torch.no_grad():
                vs, _ = E(fixed_tokens)
                vw = mapping_net(fixed_z)
                # Save both G and G_ema samples
                ema_samples = G_ema(vs, vw) * 0.5 + 0.5
                save_image(ema_samples, f"stylegan_attn_samples/ema_step_{step}.png", nrow=4)

        # ─── CHECKPOINT ───
        if step % CHECKPOINT_INTERVAL == 0 or step == TOTAL_TARGET_STEPS:
            cp = f"stylegan_attn_checkpoints/hybrid_step_{step}.pt"
            torch.save({
                'step': step,
                'generator_state_dict': G.state_dict(),
                'ema_state_dict': G_ema.state_dict(),
                'discriminator_state_dict': D.state_dict(),
                'encoder_state_dict': E.state_dict(),
                'mapping_state_dict': mapping_net.state_dict(),
                'recognizer_state_dict': R.state_dict(),
                'opt_G_state_dict': opt_G.state_dict(),
                'opt_D_state_dict': opt_D.state_dict(),
                'opt_R_state_dict': opt_R.state_dict(),
                'char_to_idx': char_to_idx,
                'idx_to_char': idx_to_char,
                'vocab': vocab
            }, cp)
            print(f"📑 Checkpoint saved: {cp}")
            
            # Purge old checkpoints to save disk (keep only latest 2)
            all_ckpts = sorted(
                [f for f in os.listdir("stylegan_attn_checkpoints") if f.startswith("hybrid_step_")],
                key=lambda x: int(x.split('_')[2].split('.')[0])
            )
            for old_ckpt in all_ckpts[:-1]:
                old_path = os.path.join("stylegan_attn_checkpoints", old_ckpt)
                os.remove(old_path)
                print(f"   🗑️ Purged old checkpoint: {old_ckpt}")

        step += 1

# ─── Final emergency save if stopped by time ───
if step > start_step:
    final_step = step - 1
    cp = f"stylegan_attn_checkpoints/hybrid_step_{final_step}.pt"
    if not os.path.exists(cp):
        torch.save({
            'step': final_step,
            'generator_state_dict': G.state_dict(),
            'ema_state_dict': G_ema.state_dict(),
            'discriminator_state_dict': D.state_dict(),
            'encoder_state_dict': E.state_dict(),
            'mapping_state_dict': mapping_net.state_dict(),
            'recognizer_state_dict': R.state_dict(),
            'opt_G_state_dict': opt_G.state_dict(),
            'opt_D_state_dict': opt_D.state_dict(),
            'opt_R_state_dict': opt_R.state_dict(),
            'char_to_idx': char_to_idx,
            'idx_to_char': idx_to_char,
            'vocab': vocab
        }, cp)
        print(f"📑 Final emergency checkpoint: {cp}")

print(f"\n✅ Training complete. Final step: {step - 1}")
print(f"Total time: {(time.time() - start_time)/3600:.2f} hours")
print(f"Samples directory: stylegan_attn_samples/")
print(f"Checkpoints directory: stylegan_attn_checkpoints/")
